In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
def generate_polynomial_data(n_points=100, noise_std=0.2, seed=42):
    """
    Generates synthetic (x, y) data for a polynomial y = 3x^3 + 2x^2 - 4x + 1 with added Gaussian noise.
    
    Args:
        n_points (int): Number of data points to generate.
        noise_std (float): Standard deviation of the Gaussian noise.
        seed (int): Random seed for reproducibility.

    Returns:
        (x, y): Tuple of Tensors representing inputs and polynomial outputs.
    """
    # Set random seed for reproducibility
    torch.manual_seed(seed)
    
    # Generate equally spaced x values
    x = torch.linspace(-3, 3, n_points).unsqueeze(1)  # shape: (n_points, 1)

    # Define polynomial: 3x^3 + 2x^2 - 4x + 1
    y_true = 3 * x**3 + 2 * x**2 - 4 * x + 1
    
    # Add Gaussian noise
    noise = torch.randn_like(y_true) * noise_std
    
    # Final y = polynomial + noise
    y = y_true + noise
    
    return x, y


In [3]:
x, y = generate_polynomial_data(n_points=1000, noise_std=0.5)

In [4]:
def generate_polynomial_data_new(n_points=100, noise_std=0.2, seed=42):
    """
    Generates synthetic (x, y) data for a polynomial y = 3x^3 + 2x^2 - 4x + 2 
    with added Gaussian noise.
    
    Args:
        n_points (int): Number of data points to generate.
        noise_std (float): Standard deviation of the Gaussian noise.
        seed (int): Random seed for reproducibility.

    Returns:
        (x, y): Tuple of Tensors representing inputs and polynomial outputs.
    """
    # Set random seed for reproducibility
    torch.manual_seed(seed)
    
    # Generate equally spaced x values
    x = torch.linspace(-3, 3, n_points).unsqueeze(1)  # shape: (n_points, 1)

    # Define polynomial: 3x^3 + 2x^2 - 4x + 2
    y_true = 3 * x**3 + 2 * x**2 - 4 * x + 2
    
    # Add Gaussian noise
    noise = torch.randn_like(y_true) * noise_std
    
    # Final y = polynomial + noise
    y = y_true + noise
    
    return x, y

In [5]:
x_new, y_new = generate_polynomial_data_new(n_points=1000, noise_std=0.75)

In [6]:
class ThreeLayerNet(nn.Module):
    def __init__(self):
        super(ThreeLayerNet, self).__init__()
        # Define a simple feed-forward network:
        # Input -> Hidden1 -> Hidden2 -> Output
        self.net = nn.Sequential(
            nn.Linear(1, 16),   # Input is 1-D -> 16 neurons
            nn.ReLU(),
            nn.Linear(16, 16),  # 16 neurons -> 16 neurons
            nn.ReLU(),
            nn.Linear(16, 1)    # 16 neurons -> 1-D output
        )
    
    def forward(self, x):
        return self.net(x)

In [7]:
model = ThreeLayerNet()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.02)

In [8]:
# Training loop
for epoch in range(100):
    optimizer.zero_grad()
    y_pred = model(x)

    loss = criterion(y_pred, y)

    loss.backward()
    optimizer.step()
    #print(f'loss: {loss.item()}')

Online learning

In [9]:
y_pred = model(x)
loss = criterion(y_pred, y)
print(f'loss: {loss.item()}')

loss: 19.194278717041016


In [10]:
y_pred = model(x_new)
loss = criterion(y_pred, y_new)
print(f'loss: {loss.item()}')

loss: 20.4167423248291


In [11]:
class OModel_param(nn.Module):
    def __init__(self, model: nn.Module, lr=0.001):
        super(OModel, self).__init__()
        self.model = model
        self.criterion = nn.MSELoss()
        
        # We'll store a learning rate for manual gradient descent
        self.lr = lr

    def forward(self, x: torch.Tensor, y_prev: torch.Tensor) -> torch.Tensor:
        """
        Perform a forward pass, compute the loss, do manual gradient descent,
        then return the updated model's prediction on x.
        
        x: current input
        y_prev: label/target you want to train on (same shape as model(x))
        """
        # 1. Zero out any existing gradients in model parameters
        for param in self.model.parameters():
            if param.grad is not None:
                param.grad.zero_()

        # 2. Forward pass
        _pred = self.model(x)  # prediction for x
        _loss = self.criterion(_pred, y_prev)

        # 3. Backward pass (compute gradients)
        _loss.backward()

        # 4. Manual gradient descent step:
        with torch.no_grad():
            for param in self.model.parameters():
                param -= self.lr * param.grad

        # 5. Return the updated prediction (could also return _pred if you prefer)
        return self.model(x)

In [12]:
class OModelAutograd_1(nn.Module):
    def __init__(self, model: nn.Module, params: list, criterion=nn.MSELoss(), lr=0.001):
        """
        Initialize the optimizer-like module.

        model: The model to optimize (used for forward computation).
        params: A list of tensors representing the parameters to be updated.
        criterion: Loss function (default is MSELoss).
        lr: Learning rate for manual updates.
        """
        super(OModelAutograd, self).__init__()
        self.model = model  # Store the model for the forward pass
        self.params = params  # List of parameters (tensors with requires_grad=True)
        self.criterion = criterion
        self.lr = torch.tensor(lr, dtype=torch.float32)  # Convert learning rate to Tensor

    def forward_fn(self, x):
        """
        Perform the forward pass using the provided model.
        """
        return self.model(x)

    def forward(self, x: torch.Tensor, y_prev: torch.Tensor) -> torch.Tensor:
        """
        Perform a forward pass, compute the loss, manually update the parameters,
        and return the updated prediction.

        x: Input tensor
        y_prev: Target tensor
        """
        # 1. Forward pass
        pred = self.forward_fn(x)  # Call the internal forward function
        loss = self.criterion(pred, y_prev)

        # 2. Compute gradients for the given parameters
        grads = torch.autograd.grad([loss], self.params, create_graph=False, retain_graph=False)

        # 3. Manual gradient descent step
        with torch.no_grad():
            for param, grad in zip(self.params, grads):
                param -= self.lr * grad  # Multiply Tensor (self.lr) with grad

        # 4. Return the updated prediction
        return self.forward_fn(x)


In [13]:
class OModelAutograd(nn.Module):
    def __init__(self, model: nn.Module, params: list, criterion=nn.MSELoss(), lr=0.001):
        """
        Initialize the optimizer-like module.

        model: The model to optimize (used for forward computation).
        params: A list of tensors representing the parameters to be updated.
        criterion: Loss function (default is MSELoss).
        lr: Learning rate for manual updates.
        """
        super(OModelAutograd, self).__init__()
        self.model = model  # Store the model for the forward pass
        self.params = [p for p in params if p.requires_grad]  # Filter parameters requiring gradients
        self.criterion = criterion
        self.lr = torch.tensor(lr, dtype=torch.float32)  # Convert learning rate to Tensor

    def forward_fn(self, x):
        """
        Perform the forward pass using the provided model.
        """
        return self.model(x)

    def forward(self, x: torch.Tensor, y_prev: torch.Tensor) -> torch.Tensor:
        """
        Perform a forward pass, compute the loss, manually update the parameters,
        and return the updated prediction.

        x: Input tensor
        y_prev: Target tensor
        """
        # 1. Forward pass
        pred = self.forward_fn(x)  # Call the internal forward function
        loss = self.criterion(pred, y_prev)

        # 2. Compute gradients for the given parameters
        grads = torch.autograd.grad(
            [loss], self.params, create_graph=False, retain_graph=False, allow_unused=True
        )

        # 3. Manual gradient descent step
        with torch.no_grad():
            for param, grad in zip(self.params, grads):
                if grad is not None:  # Handle None gradients
                    param -= self.lr * grad

        # 4. Return the updated prediction
        return self.forward_fn(x)


In [14]:
new_model = OModelAutograd(
    model = model,
    params=list(model.parameters()),  # Pass the model's parameters explicitly
    criterion=nn.MSELoss(),  # Specify the loss function
    lr=0.001  # Learning rate
)

# Example training loop
for _ in range(100):
    y_pred = new_model(x, y)  # Forward pass, loss calculation, and parameter update
    loss = new_model.criterion(y_pred, y)  # Optional: track the loss
    print(f'Loss: {loss.item()}')

Loss: 18.879966735839844
Loss: 18.597335815429688
Loss: 18.32732391357422
Loss: 18.06534767150879
Loss: 17.810346603393555
Loss: 17.561569213867188
Loss: 17.31809425354004
Loss: 17.080183029174805
Loss: 16.847248077392578
Loss: 16.619304656982422
Loss: 16.396780014038086
Loss: 16.17939567565918
Loss: 15.966535568237305
Loss: 15.757963180541992
Loss: 15.553585052490234
Loss: 15.353001594543457
Loss: 15.155622482299805
Loss: 14.961516380310059
Loss: 14.77092456817627
Loss: 14.583280563354492
Loss: 14.39843463897705
Loss: 14.21656608581543
Loss: 14.037686347961426
Loss: 13.861503601074219
Loss: 13.688146591186523
Loss: 13.51745319366455
Loss: 13.34910774230957
Loss: 13.18343448638916
Loss: 13.020441055297852
Loss: 12.859827995300293
Loss: 12.701872825622559
Loss: 12.546728134155273
Loss: 12.394142150878906
Loss: 12.243742942810059
Loss: 12.095525741577148
Loss: 11.949390411376953
Loss: 11.805224418640137
Loss: 11.66285514831543
Loss: 11.522323608398438
Loss: 11.383840560913086
Loss: 11.24

Save as script

In [16]:
script_model = torch.jit.script(new_model)
script_model.save(f"./online_example.pt")
